# Agentic w/ filesystem tools + delegate task

Agentic search over WANDS using filesystem tools (ls/grep/cat).

ELI5: we turn the product catalog into a folder of text files. The agent can only list files (ls), search inside files (grep), and open files (cat). When a task is too big, it can ask a helper agent to do a mini-search using the same tools.

This notebook mirrors `configs/cheat-at-search/agentic_wands_fs_tools_delegate.yml` with:
- model: gpt-5-mini
- tools: ls_wands, grep_wands, cat_wands, delegate_task
- validator: require at least 10 results


In [ ]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git@ee2526eb8bfac087dc3f90522cc7191032e47dfd
from cheat_at_search.data_dir import mount
try:
    mount(use_gdrive=True)
except ImportError:
    from pathlib import Path
    manual_path = str(Path.home() / ".search-experiments" / "cheat-at-search")
    mount(use_gdrive=False, manual_path=manual_path)

## Get an OpenAI Key + load corpus

This will prompt you for an OpenAI Key to interact with GPT-5.

In [ ]:
import logging
import numpy as np
import pandas as pd

from openai import OpenAI
from cheat_at_search.data_dir import key_for_provider
from cheat_at_search.wands_data import corpus, judgments

OPENAI_KEY = key_for_provider("openai")
openai = OpenAI(api_key=OPENAI_KEY)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("agentic_fs_delegate")

corpus = corpus.reset_index(drop=True)
doc_id_lookup = corpus["doc_id"].to_numpy()
doc_id_to_index = {doc_id: idx for idx, doc_id in enumerate(doc_id_lookup)}

corpus[["doc_id", "title", "description"]].head(3)

## Sample queries (8-16)

We keep the notebook fast by sampling a small set of queries with a fixed seed.

In [ ]:
QUERY_COUNT = 12
queries = judgments[["query", "query_id"]].drop_duplicates()
queries = queries.sample(n=QUERY_COUNT, random_state=7).reset_index(drop=True)
queries

## Build virtual file system paths and contents

ELI5: each product becomes a pretend text file stored in folders by category.

In [ ]:
import hashlib
import re

def _slugify(value: str) -> str:
    lowered = value.lower()
    slug = re.sub(r"[^a-z0-9]+", "-", lowered)
    slug = re.sub(r"-+", "-", slug).strip("-")
    return slug

def _slugify_series(values: pd.Series) -> pd.Series:
    return (
        values.str.lower()
        .str.replace(r"[^a-z0-9]+", "-", regex=True)
        .str.replace(r"-+", "-", regex=True)
        .str.strip("-")
    )

def _build_filename(title_slug: str, id_slug: str, *, max_len: int = 200) -> str:
    base = f"{title_slug}-{id_slug}.txt"
    if len(base) <= max_len:
        return base
    digest = hashlib.sha1(base.encode("utf-8")).hexdigest()[:8]
    suffix = f"-{digest}-{id_slug}.txt"
    max_title_len = max_len - len(suffix)
    if max_title_len < 1:
        truncated_title = title_slug[:1] if title_slug else "d"
        return f"{truncated_title}{suffix}"
    truncated_title = title_slug[:max_title_len]
    return f"{truncated_title}{suffix}"

def _wands_path_builder(title_series: pd.Series, doc_id_series: pd.Series, corpus_df: pd.DataFrame) -> pd.Series:
    title_slug = _slugify_series(title_series).mask(lambda s: s == "", "document")
    id_slug = _slugify_series(doc_id_series).mask(lambda s: s == "", doc_id_series)
    base_name = pd.Series(
        [
            _build_filename(title, doc_id)
            for title, doc_id in zip(title_slug.tolist(), id_slug.tolist())
        ],
        index=title_series.index,
    )

    category_series = corpus_df.get("category")
    if category_series is None:
        category_series = pd.Series("", index=corpus_df.index)
    category_series = category_series.fillna("").astype(str)
    category_slug = _slugify_series(category_series)

    subcategory_series = corpus_df.get("subcategory")
    if subcategory_series is None and "sub_category" in corpus_df.columns:
        subcategory_series = corpus_df.get("sub_category")
    if subcategory_series is None:
        subcategory_series = pd.Series("", index=corpus_df.index)
    subcategory_series = subcategory_series.fillna("").astype(str)
    subcategory_slug = _slugify_series(subcategory_series)

    path = "/" + base_name
    has_category = category_slug != ""
    has_subcategory = subcategory_slug != ""
    path = path.where(~has_category, "/" + category_slug + "/" + base_name)
    path = path.where(
        ~(has_category & has_subcategory),
        "/" + category_slug + "/" + subcategory_slug + "/" + base_name,
    )
    return path

def ensure_wands_paths(corpus_df: pd.DataFrame) -> pd.DataFrame:
    corpus_df = corpus_df.copy()
    if "path" in corpus_df.columns or "contents" in corpus_df.columns:
        raise ValueError("Corpus already has path/contents columns.")
    title_series = corpus_df.get("title")
    if title_series is None:
        title_series = pd.Series("", index=corpus_df.index)
    title_series = title_series.fillna("").astype(str)

    description_series = corpus_df.get("description")
    if description_series is None:
        description_series = pd.Series("", index=corpus_df.index)
    description_series = description_series.fillna("").astype(str)

    doc_id_series = corpus_df.get("doc_id")
    if doc_id_series is None:
        doc_id_series = pd.Series(corpus_df.index, index=corpus_df.index)
    doc_id_series = doc_id_series.fillna("").astype(str)

    path_series = _wands_path_builder(title_series, doc_id_series, corpus_df)
    corpus_df["path"] = path_series
    corpus_df["contents"] = (
        title_series
        + " (ID: "
        + doc_id_series
        + ")\n\n"
        + description_series
    )
    return corpus_df

corpus_fs = ensure_wands_paths(corpus)
corpus_fs[["path", "contents"]].head(2)

## Filesystem tool factory

This is the heart of the virtual file system. We create three tools:
- `ls_wands`: list directories and files
- `grep_wands`: search for text inside files
- `cat_wands`: open a file and read its contents

ELI5: we are giving the agent a tiny terminal. It cannot see the raw dataframe. It only gets these three commands.

In [ ]:
from pathlib import PurePosixPath

FS_STATE = {}

def _normalize_dir(path: str) -> str:
    if not isinstance(path, str):
        raise ValueError("path must be a string")
    trimmed = path.strip() or "/"
    if not trimmed.startswith("/"):
        trimmed = f"/{trimmed}"
    if trimmed != "/" and not trimmed.endswith("/"):
        trimmed = f"{trimmed}/"
    return trimmed

def _normalize_glob(prefix: str, glob: str) -> str:
    if not isinstance(glob, str):
        raise ValueError("glob must be a string")
    if not glob:
        glob = "*"
    if glob.startswith("/"):
        return glob
    return f"{prefix}{glob}"

def _normalize_match_pattern(pattern: str) -> str:
    normalized = pattern.strip()
    if normalized.startswith("/"):
        normalized = normalized[1:]
    return normalized

def _normalize_path(path: str) -> str:
    trimmed = path.strip().strip("\"").strip("'")
    if not trimmed:
        return ""
    if trimmed.startswith("./"):
        trimmed = trimmed[2:]
    if not trimmed.startswith("/"):
        trimmed = f"/{trimmed}"
    while "//" in trimmed:
        trimmed = trimmed.replace("//", "/")
    return trimmed

def _match_glob(pattern: str, path: str) -> bool:
    normalized_pattern = _normalize_match_pattern(pattern)
    normalized_path = path[1:] if path.startswith("/") else path
    path_obj = PurePosixPath(normalized_path)
    if path_obj.match(normalized_pattern):
        return True
    if normalized_pattern.startswith("**/") and "/" not in normalized_path:
        return path_obj.match(normalized_pattern[3:])
    return False

def _snippet_from_match(text: str, match: re.Match, window: int = 60) -> str:
    start = max(0, match.start() - window)
    end = min(len(text), match.end() + window)
    snippet = text[start:end]
    snippet = " ".join(snippet.splitlines()).strip()
    return snippet

def make_wands_fs_tools(corpus_df: pd.DataFrame):
    FS_STATE["path_series"] = corpus_df["path"].astype(str)
    FS_STATE["contents_series"] = corpus_df["contents"].astype(str)
    FS_STATE["paths"] = FS_STATE["path_series"].tolist()
    FS_STATE["path_contents"] = list(zip(FS_STATE["paths"], FS_STATE["contents_series"].tolist()))

    def ls_wands(path: str, glob: str, max_results: int = 50, agent_state=None) -> list[str] | str:
        """List files in a directory matching the glob, at most 50 results. Returns a list of paths.

        WANDS filesystem layout uses <category>/<subcategory>/<product-name-slug>-<doc-id>.txt. Example: /Furniture/Armchairs/sancroft-armchair-1234.txt. File contents are:

        <Title> (ID: <ID>)

        <Description>

        Example file contents:

        Sancroft Armchair (ID: 1234)

        A compact armchair with tailored upholstery, a supportive back, and gently flared arms designed for small spaces.
        """
        limit = max_results
        if max_results > 50:
            limit = 50
        if max_results <= 0:
            return []
        if not isinstance(path, str) or not path.strip():
            return "Error! path must be a non-empty string."
        prefix = _normalize_dir(path)
        if glob in {"*", "*/"}:
            child_map = {}
            for item in FS_STATE["paths"]:
                if not item.startswith(prefix):
                    continue
                rest = item[len(prefix):]
                if not rest:
                    continue
                child = rest.split("/", 1)[0]
                child_path = f"{prefix}{child}" if prefix != "/" else f"/{child}"
                is_dir = "/" in rest
                if child_path in child_map:
                    child_map[child_path] = child_map[child_path] or is_dir
                else:
                    child_map[child_path] = is_dir
            dir_children = sorted([path for path, is_dir in child_map.items() if is_dir])
            file_children = sorted([path for path, is_dir in child_map.items() if not is_dir])
            ordered = dir_children + file_children
            if len(ordered) > limit:
                extra = len(ordered) - limit
                ordered = ordered[:limit]
                ordered.append(f"Truncated ({extra} more)")
            return ordered
        pattern = _normalize_glob(prefix, glob)
        matches = []
        extra = 0
        for item in FS_STATE["paths"]:
            if not item.startswith(prefix):
                continue
            if _match_glob(pattern, item):
                if len(matches) < limit:
                    matches.append(item)
                else:
                    extra += 1
        matches = sorted(matches)
        if extra:
            matches.append(f"Truncated ({extra} more)")
        return matches

    def grep_wands(pattern: str, glob: str, num_results: int = 50, agent_state=None) -> list[dict[str, str]] | str:
        """Search for a regex pattern in files matching the glob, at most 50 results.

        WANDS filesystem layout uses <category>/<subcategory>/<product-name-slug>-<doc-id>.txt. Example: /Furniture/Armchairs/sancroft-armchair-1234.txt. File contents are:

        <Title> (ID: <ID>)

        <Description>

        Example file contents:

        Sancroft Armchair (ID: 1234)

        A compact armchair with tailored upholstery, a supportive back, and gently flared arms designed for small spaces.
        """
        limit = num_results
        if num_results > 50:
            limit = 50
        if num_results <= 0:
            return []
        try:
            regex = re.compile(pattern)
        except re.error:
            return f"Error! Invalid regex pattern: {pattern}"
        match_pattern = _normalize_glob("/", glob)
        results = []
        extra = 0
        for path, contents in FS_STATE["path_contents"]:
            if not _match_glob(match_pattern, path):
                continue
            match = regex.search(contents)
            if not match:
                continue
            if len(results) < limit:
                results.append({"path": path, "snippet": _snippet_from_match(contents, match)})
            else:
                extra += 1
        if extra:
            results.append({"path": "", "snippet": f"Truncated ({extra} more)"})
        return results

    def cat_wands(path: str, agent_state=None) -> str:
        """Return the contents of a file as a string.

        WANDS filesystem layout uses <category>/<subcategory>/<product-name-slug>-<doc-id>.txt. Example: /Furniture/Armchairs/sancroft-armchair-1234.txt. File contents are:

        <Title> (ID: <ID>)

        <Description>

        Example file contents:

        Sancroft Armchair (ID: 1234)

        A compact armchair with tailored upholstery, a supportive back, and gently flared arms designed for small spaces.
        """
        if not isinstance(path, str) or not path.strip():
            return "Error! path must be a non-empty string."
        normalized_path = _normalize_path(path)
        if not normalized_path:
            return "Error! path must be a non-empty string."
        matches = FS_STATE["contents_series"][FS_STATE["path_series"] == normalized_path]
        if matches.empty and normalized_path != path:
            matches = FS_STATE["contents_series"][FS_STATE["path_series"] == path]
        if matches.empty:
            filename = PurePosixPath(normalized_path or path).name
            if filename:
                filename_matches = FS_STATE["contents_series"][FS_STATE["path_series"].str.endswith(f"/{filename}")]
                if len(filename_matches) == 1:
                    return str(filename_matches.iloc[0])
                if len(filename_matches) > 1:
                    return f"Error! Multiple files found for filename: {filename}"
            return f"Error! No file found for path: {path}"
        if len(matches) > 1:
            return f"Error! Multiple files found for path: {path}"
        return str(matches.iloc[0])

    return [ls_wands, grep_wands, cat_wands]

tools = make_wands_fs_tools(corpus_fs)
ls_tool, grep_tool, cat_tool = tools
ls_tool("/", "*", max_results=5)

## Delegate task tool

ELI5: sometimes the main agent wants a helper to do a mini-search. `delegate_task` spins up a sub-agent with the same tools and returns any tool outputs it finds.

In [ ]:
import json

from cheat_at_search.agent.openai_agent import OpenAIAgent

def _append_tool_output(results, output):
    if isinstance(output, str):
        try:
            output = json.loads(output)
        except json.JSONDecodeError:
            return
    if isinstance(output, list):
        for item in output:
            if isinstance(item, dict):
                results.append(item)
        return
    if isinstance(output, dict):
        results.append(output)

def _collect_tool_outputs(items):
    results = []
    for item in items:
        if not isinstance(item, dict):
            continue
        if item.get("type") != "function_call_output":
            continue
        _append_tool_output(results, item.get("output"))
    return results

def build_delegate_task(search_tools, model, reasoning, system_prompt):
    filtered_tools = [tool for tool in search_tools if getattr(tool, "__name__", "") != "delegate_task"]

    def delegate_task(task: str, agent_state=None):
        """Delegate a search task to a sub-agent and return tool results."""
        if agent_state is None:
            agent_state = {}
        depth = agent_state.get("delegate_task_depth", 0) + 1
        if depth > 1:
            return "Error! delegate_task cannot be nested."
        agent_state["delegate_task_depth"] = depth
        try:
            agent = OpenAIAgent(
                tools=filtered_tools,
                model=f"openai/{model}" if "/" not in model else model,
                reasoning_level=reasoning,
                response_model=None,
            )
            inputs = [
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": (
                        f"Task: {task}\n"
                        "Use the available search tools to find results. "
                        "Do not ask follow-up questions; return tool results."
                    ),
                },
            ]
            previous_inputs = list(inputs)
            _, inputs, _ = agent.chat(inputs=inputs, agent_state=agent_state, logger=logger)
            new_items = inputs[len(previous_inputs):]
            results = _collect_tool_outputs(new_items)
            return results
        finally:
            agent_state["delegate_task_depth"] = depth - 1

    delegate_task.__name__ = "delegate_task"
    return delegate_task

delegate_task_tool = build_delegate_task(
    search_tools=tools,
    model="gpt-5-mini",
    reasoning="medium",
    system_prompt="You help with tasks searchinging / finding content as instructed",
)
delegate_task_tool("Find 2 file paths mentioning chair", agent_state={})[:2]

## Agentic strategy with validator

We use the config's validator: if fewer than 10 results are returned, we ask the agent to try again.

ELI5: if the agent brings back too few items, we politely tell it to keep looking.

In [ ]:
from pydantic import BaseModel, Field
from cheat_at_search.strategy import SearchStrategy

SYSTEM_PROMPT = """
You take user search queries and use filesystem tools to find relevant products.

Use ls_wands to explore directories, grep_wands to search within files, and cat_wands
to inspect specific product files.

Use delegate_task to describe a more complex search task to a helpful sub-agent that can
plan out multiple steps and use tools iteratively to accomplish the task.

It's important to return results ranked from most to least relevant based on the user query.

Gather results until you have 10 best matches you can find. It's important to return at least 10.
Return the *DOC IDs*, not paths, in the response.
"""

VALIDATOR_PROMPT = (
    "Please return at least 10 results to give the user a good variety to choose from."
)

class SearchResults(BaseModel):
    """The state of the search agent, which can be used to inform future reasoning and tool use."""
    ranked_results: list[str] = Field(description="Top ranked search results (their doc_ids) when complete")

def _needs_more_results(resp, min_results=10):
    if resp is None:
        return True
    ranked = resp.ranked_results or []
    return len(ranked) < min_results

class AgenticFilesystemDelegateStrategy(SearchStrategy):
    def __init__(self, corpus_df, tools, model="gpt-5-mini", workers=1, max_loops=3):
        self.tools = tools
        self.model = model
        self.max_loops = max_loops
        super().__init__(corpus_df, workers=workers)

    def search(self, query: str, k: int = 10):
        inputs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"The user's query: {query}"},
        ]
        agent_state = {"trace_logger": logger, "num_tool_calls": 0}
        agent = OpenAIAgent(
            tools=self.tools,
            model=f"openai/{self.model}" if "/" not in self.model else self.model,
            response_model=SearchResults,
            reasoning_level="medium",
        )
        resp = None
        for _ in range(self.max_loops):
            resp, inputs, _ = agent.chat(inputs=inputs, agent_state=agent_state, logger=logger)
            if not _needs_more_results(resp, min_results=10):
                break
            inputs.append({"role": "user", "content": VALIDATOR_PROMPT})

        ranked = [doc_id for doc_id in (resp.output_parsed.ranked_results or [])] if resp else []
        ranked = [doc_id_to_index.get(int(doc_id), -1) for doc_id in ranked if str(doc_id).isdigit()]
        ranked = [idx for idx in ranked if idx >= 0][:k]
        return ranked, [1.0] * len(ranked)

delegate_tools = [ls_tool, grep_tool, cat_tool, delegate_task_tool]
strategy = AgenticFilesystemDelegateStrategy(corpus_fs, delegate_tools, workers=1)
strategy.search(queries.loc[0, "query"], k=5)

## Run the small benchmark

We run `run_strategy` on a small subset of queries and compute the mean NDCG.

ELI5: NDCG is a score from 0 to 1 that says how good the ranking is (higher is better).

In [ ]:
from cheat_at_search.search import run_strategy, ndcgs

results = run_strategy(strategy, judgments, num_queries=QUERY_COUNT, seed=7, cache=False)
ndcg_series = ndcgs(results)
pd.DataFrame({"metric": ["NDCG@10"], "value": [ndcg_series.mean()]})